# 🛠️ Oficina Digital: Gemini Notebook + Graphify + Automação Python
### Do Chamado Bruto ao Diagnóstico Inteligente com Grafo de Conhecimento

Bem-vindo ao laboratório prático da **Oficina Digital**!

Neste roteiro, você vai transformar uma assistência técnica tradicional em um sistema autônomo com **memória institucional**. Em vez de apenas classificar chamados de forma isolada, nosso sistema vai:

1. **Extrair entidades técnicas com o Google Gemini** (marca, modelo, sintomas, componentes suspeitos).
2. **Construir um Grafo de Conhecimento (a mesma ideia do Graphify)** com histórico de falhas, peças e soluções anteriores.
3. **Aplicar GraphRAG**: usar a memória do grafo para alimentar o Gemini com diagnósticos anteriores reais da oficina.
4. **Automatizar o fluxo em Python** diretamente com o banco de dados (`SQLite`) da Oficina Digital.

---
### 📋 Sumário
- **Passo 1:** Configuração do Ambiente e Chave do Gemini
- **Passo 2:** Extração Estruturada com Gemini API (`google-genai` + Pydantic)
- **Passo 3:** Modelando a Memória da Oficina com Grafo de Conhecimento
- **Passo 4:** GraphRAG (Diagnóstico Inteligente baseado em Casos Anteriores)
- **Passo 5:** Automação em Python com o Banco da Oficina (`chamados.db`)
- **Passo 6:** Desafio Prático para o Aluno

## 📦 Passo 1: Configuração do Ambiente

Execute a célula abaixo para instalar as bibliotecas necessárias. Usaremos o SDK oficial **`google-genai`**, o **`pydantic`** para validação estruturada e o **`networkx`** com **`matplotlib`** para manipular e visualizar o grafo de conhecimento (a mesma ideia que o **Graphify** usa para montar o grafo de uma pasta).

In [ ]:
# Instalação dos pacotes necessários
!pip install -q -U google-genai pydantic networkx matplotlib

### Configurando a API Key do Google Gemini
Você pode obter sua chave gratuitamente no [Google AI Studio](https://aistudio.google.com/).

Defina sua chave abaixo (ou adicione às variáveis de ambiente):

In [ ]:
import os
from google import genai
from google.genai import types

# Se estiver rodando no Google Colab, pode usar userdata.get('GEMINI_API_KEY')
api_key = os.environ.get("GEMINI_API_KEY", "SUA_CHAVE_GEMINI_AQUI")

client = genai.Client(api_key=api_key)
print("✅ Cliente Gemini inicializado com sucesso!")

## 🧠 Passo 2: Extração Estruturada com Gemini e Pydantic

Quando um cliente abre um chamado pelo site, o texto muitas vezes é informal ou confuso:
> *"Meu notebook Dell G15 desliga sozinho do nada quando abro o Valorant ou renderizo vídeo, tá esquentando que dá pra fritar ovo."*

O primeiro trabalho da IA é **transformar texto livre em dados estruturados** para alimentar nosso banco e nosso grafo.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ExtracaoChamado(BaseModel):
    marca: str = Field(description="Marca do equipamento, ex: Dell, Lenovo, Apple, Samsung")
    modelo: str = Field(description="Modelo específico se informado ou categoria, ex: G15, Nitro 5, Desktop")
    tipo_dispositivo: str = Field(description="notebook, desktop, all-in-one ou servidor")
    sintomas: List[str] = Field(description="Lista de sintomas observados, ex: [superaquecimento, desligamento repentino]")
    componentes_suspeitos: List[str] = Field(description="Componentes com provável falha, ex: [pasta termica, cooler, fonte, ssd]")
    gravidade: str = Field(description="alta, media ou baixa")

def extrair_entidades(relato_cliente: str, equipamento_informado: str) -> ExtracaoChamado:
    prompt = f"""
    Você é um especialista em triagem de hardware.
    Analise o equipamento e o relato do cliente e extraia os dados técnicos com precisão.
    
    Equipamento informado pelo cliente: {equipamento_informado}
    Relato do problema: {relato_cliente}
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ExtracaoChamado,
            temperature=0.1
        )
    )
    return ExtracaoChamado.model_validate_json(response.text)

# Teste da extração
relato_teste = "Meu notebook Dell G15 desliga sozinho do nada quando abro o Valorant ou renderizo vídeo, tá esquentando que dá pra fritar ovo."
dados = extrair_entidades(relato_teste, "Dell G15")
print("Resultado estruturado pelo Gemini:")
print(dados.model_dump_json(indent=2))

## 🕸️ Passo 3: O que é Graphify e por que usar Grafos de Conhecimento?

### O Problema do RAG Tradicional (Vetorial puro):
- Bancos vetoriais encontram textos com palavras parecidas, mas **não compreendem relações estruturadas** como:
  - `(Dell G15) -[APRESENTA]-> (Superaquecimento) -[RESOLVEU_COM]-> (Troca de Pasta Térmica + Limpeza)`
  - `(Troca de Pasta Térmica) -[UTILIZA_PEÇA]-> (Pasta Térmica Alta Condutividade)`

### Como o Graphify funciona:
- O **Graphify** lê uma pasta de arquivos (documentos, código, imagens) e monta um **grafo de conhecimento** navegável, com grupos de assuntos e uma trilha que separa o que foi extraído do que foi inferido. Ele também exporta para o Obsidian e responde perguntas (`/graphify query`).
- Vamos implementar a estrutura do Grafo de Conhecimento da Oficina e povoá-lo com casos históricos de bancada!

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

class GrafoConhecimentoOficina:
    """Simulador do Grafo de Conhecimento da Oficina (mesma ideia do Graphify)."""
    def __init__(self):
        self.grafo = nx.DiGraph()

    def adicionar_caso_historico(self, modelo: str, sintoma: str, causa: str, peca: str, taxa_sucesso: float = 1.0):
        # Adiciona nós com tipos
        self.grafo.add_node(modelo, tipo="Modelo")
        self.grafo.add_node(sintoma, tipo="Sintoma")
        self.grafo.add_node(causa, tipo="Causa")
        self.grafo.add_node(peca, tipo="Peca")

        # Adiciona arestas relacionais
        self.grafo.add_edge(modelo, sintoma, relacao="apresenta")
        self.grafo.add_edge(sintoma, causa, relacao="provocado_por")
        self.grafo.add_edge(causa, peca, relacao="requer_troca_de", taxa_sucesso=taxa_sucesso)

    def buscar_antecedentes(self, modelo: str, sintomas: list[str]) -> list[dict]:
        resultados = []
        for sintoma in sintomas:
            # Busca se há caminhos no grafo ligando modelo -> sintoma -> causa -> peca
            for n_modelo in self.grafo.nodes:
                if modelo.lower() in n_modelo.lower() or n_modelo.lower() in modelo.lower():
                    for viz_sintoma in self.grafo.neighbors(n_modelo):
                        if sintoma.lower() in viz_sintoma.lower() or viz_sintoma.lower() in sintoma.lower():
                            for causa in self.grafo.neighbors(viz_sintoma):
                                pecas = list(self.grafo.neighbors(causa))
                                resultados.append({
                                    "modelo": n_modelo,
                                    "sintoma": viz_sintoma,
                                    "causa": causa,
                                    "pecas": pecas
                                })
        return resultados

# Inicializa o grafo e popula com o histórico real da bancada da oficina
memoria_oficina = GrafoConhecimentoOficina()

memoria_oficina.adicionar_caso_historico(
    modelo="Dell G15",
    sintoma="desligamento repentino",
    causa="pasta térmica de fábrica ressecada e aletas do cooler obstruídas",
    peca="Pasta Térmica Prata + Pad Térmico"
)

memoria_oficina.adicionar_caso_historico(
    modelo="Dell G15",
    sintoma="não liga",
    causa="curto no circuito de charge (MOSFET de entrada)",
    peca="MOSFET Canal N 30V + Fonte 130W"
)

memoria_oficina.adicionar_caso_historico(
    modelo="Acer Nitro 5",
    sintoma="tela azul intermitente",
    causa="oxidação nos contatos da memória RAM DDR4",
    peca="Limpa Contato Isopropílico + Memória DDR4 8GB"
)

memoria_oficina.adicionar_caso_historico(
    modelo="Desktop Genérico",
    sintoma="reinicia ao abrir jogos",
    causa="queda de tensão na linha 12V da fonte sob carga pesada",
    peca="Fonte ATX 600W 80 Plus"
)

print("✅ Grafo de Conhecimento da Oficina criado com sucesso!")
print(f"Nós no grafo: {memoria_oficina.grafo.number_of_nodes()} | Relações: {memoria_oficina.grafo.number_of_edges()}")

### Visualizando o Grafo de Conhecimento
Execute a célula abaixo para ver o mapa visual de conexões da oficina:

In [ ]:
plt.figure(figsize=(12, 7))
pos = nx.spring_layout(memoria_oficina.grafo, seed=42, k=1.2)

cores = []
for no, attrs in memoria_oficina.grafo.nodes(data=True):
    t = attrs.get("tipo", "")
    if t == "Modelo": cores.append("#4A90E2")      # Azul
    elif t == "Sintoma": cores.append("#F5A623")   # Amarelo/Laranja
    elif t == "Causa": cores.append("#D0021B")     # Vermelho
    elif t == "Peca": cores.append("#7ED321")      # Verde
    else: cores.append("#9B9B9B")

nx.draw_networkx_nodes(memoria_oficina.grafo, pos, node_color=cores, node_size=2000, alpha=0.9)
nx.draw_networkx_labels(memoria_oficina.grafo, pos, font_size=8, font_weight="bold")
nx.draw_networkx_edges(memoria_oficina.grafo, pos, edge_color="gray", arrows=True, arrowsize=15)

plt.title("Grafo de Conhecimento da Oficina Digital", fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()

## 🚀 Passo 4: GraphRAG com Gemini

Agora unimos os dois mundos:
1. O cliente envia um chamado.
2. O Gemini extrai as entidades.
3. O sistema consulta o **Grafo de Conhecimento** procurando casos anteriores idênticos ou parecidos.
4. Se houver histórico no Grafo, o Gemini usa essa **experiência prévia** para gerar um diagnóstico de alta precisão!

Veja a diferença: sem o grafo, o modelo daria respostas genéricas de internet ("pode ser vírus ou software"). Com o Grafo, ele já sabe exatamente o defeito crônico daquele modelo!

In [ ]:
def diagnosticar_com_graphrag(relato: str, equipamento: str, grafo: GrafoConhecimentoOficina):
    # 1. Extração estruturada
    extracao = extrair_entidades(relato, equipamento)
    
    # 2. Busca de antecedentes no Grafo
    antecedentes = grafo.buscar_antecedentes(extracao.modelo, extracao.sintomas)
    
    contexto_grafo = ""
    if antecedentes:
        contexto_grafo = "HISTÓRICO DA BANCADA DA OFICINA (GRAFO DE CONHECIMENTO):\n"
        for i, a in enumerate(antecedentes, 1):
            contexto_grafo += f"- Caso {i}: Modelo '{a['modelo']}', Sintoma '{a['sintoma']}' -> Causa confirmada: '{a['causa']}' -> Peças usadas: {', '.join(a['pecas'])}\n"
    else:
        contexto_grafo = "Nenhum caso anterior idêntico registrado no grafo da oficina. Usar raciocínio padrão de bancada."

    # 3. Prompt do GraphRAG para o Gemini
    prompt_diagnostico = f"""
    Você é o Técnico Chefe da Oficina Digital.
    Gere o diagnóstico técnico e as orientações para o técnico que irá atender na bancada.

    CHAMADO DO CLIENTE:
    Equipamento: {equipamento}
    Relato: {relato}

    DADOS EXTRAÍDOS:
    Marca/Modelo: {extracao.marca} {extracao.modelo}
    Sintomas: {', '.join(extracao.sintomas)}
    Gravidade: {extracao.gravidade}

    {contexto_grafo}

    RESPONDA EM FORMATO CLARO COM OS TÓPICOS:
    1. Causa Provável (levando em conta o histórico do grafo, se houver)
    2. Testes de Bancada Recomendados (passo a passo para o técnico)
    3. Peças / Insumos Prováveis
    4. Mensagem Transparente para o Cliente (sem prometer valor ou prazo fixo)
    """
    
    resposta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt_diagnostico
    )
    return resposta.text

# Teste com um chamado real que acionará o grafo
chamado_cliente = "Meu Dell G15 esquenta demais e simplesmente desliga no meio do jogo. O cooler parece fazer barulho estranho."
resultado = diagnosticar_com_graphrag(chamado_cliente, "Dell G15", memoria_oficina)
print("=== DIAGNÓSTICO ENRIQUECIDO COM GRAPHRAG ===")
print(resultado)

## 🤖 Passo 5: Automação em Python com o Banco de Dados (`chamados.db`)

Agora vamos construir o script de automação que conecta a API existente (`chamados.db` em SQLite) com o pipeline do Gemini + Grafo.

O script:
1. Conecta no banco SQLite da pasta `../api/chamados.db`.
2. Busca chamados que ainda estão com status `aberto` e sem diagnóstico.
3. Processa cada um automaticamente via Gemini e Grafo.
4. Atualiza o banco de dados com a prioridade e diagnóstico técnico.
5. Povoa o Grafo com o novo aprendizado.

In [ ]:
import sqlite3
from pathlib import Path

# Localiza o banco da API
# Localiza o banco da API (suporta pasta Downloads, raiz ou api/)
candidatos = [
    Path("../api/chamados.db"),
    Path("api/chamados.db"),
    Path.home() / "Downloads" / "oficina-digital" / "api" / "chamados.db",
    Path("chamados.db")
]
caminho_banco = next((p for p in candidatos if p.exists()), None)
if not caminho_banco:
    caminho_banco = Path("chamados_lab.db")
    # Cria banco local para o laboratório caso a api ainda não tenha rodado
    with sqlite3.connect(caminho_banco) as con:
        con.execute("""
            CREATE TABLE IF NOT EXISTS chamados (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT, contato TEXT, equipamento TEXT, problema TEXT,
                status TEXT DEFAULT 'aberto', prioridade TEXT, diagnostico TEXT, criado_em TEXT
            )""")
        # Insere chamado de teste
        con.execute("""
            INSERT INTO chamados (nome, contato, equipamento, problema, status, criado_em)
            VALUES ('Carlos Lima', 'carlos@email.com', 'Dell G15', 'Desliga sozinho ao abrir jogos e aquece muito', 'aberto', datetime('now'))
        """)

def executar_automacao():
    print(f"🔍 Conectando ao banco de dados: {caminho_banco}")
    with sqlite3.connect(caminho_banco) as con:
        con.row_factory = sqlite3.Row
        pendentes = con.execute("SELECT * FROM chamados WHERE status = 'aberto' AND diagnostico IS NULL").fetchall()
        
        print(f"📦 Encontrados {len(pendentes)} chamados pendentes para automação.")
        for linha in pendentes:
            print(f"\n▶️ Processando Chamado #{linha['id']} - {linha['equipamento']}")
            
            # Executa GraphRAG com Gemini
            diag_texto = diagnosticar_com_graphrag(linha['problema'], linha['equipamento'], memoria_oficina)
            
            # Atualiza banco com diagnóstico gerado
            con.execute("""
                UPDATE chamados
                SET diagnostico = ?, prioridade = 'alta'
                WHERE id = ?
            """, (diag_texto, linha['id']))
            
            print(f"✅ Chamado #{linha['id']} atualizado com diagnóstico inteligente no banco!")

executar_automacao()

## 🏆 Passo 6: Desafios e Próximos Passos para o Aluno

Parabéns! Você acabou de criar um pipeline completo de IA com **Memória Estruturada em Grafo** e **Automação**.

Aqui estão 3 desafios para você expandir o projeto:

1. **Desafio 1 (Feedback Loop)**: Crie uma função que, quando o técnico finalizar o chamado no Telegram com `/concluir ID`, leia a peça que foi realmente trocada e insira uma nova aresta no grafo automaticamente.
2. **Desafio 2 (Graphify de verdade)**: rode `/graphify` na pasta `docs/notebooklm` do projeto e abra o arquivo HTML criado na pasta `graphify-out`. Compare com o grafo que você montou à mão.
3. **Desafio 3 (Estimativa de Preço e Tempo)**: Adicione ao modelo Pydantic do Gemini campos para `tempo_estimado_horas` e `faixa_custo_estimada` para que o Telegram já envie uma estimativa orçamentária preliminar ao técnico.